<a href="https://colab.research.google.com/github/Tetbet/AQI_Predictor_BYOP/blob/main/notebooks/01_AQI_EDA_and_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

In [ ]:
# Path to the file in your Google Drive
file_path = '/content/drive/MyDrive/AQI_Project/city_day.csv'

# Load the data
df = pd.read_csv(file_path)

# Convert the 'Date' column to actual Datetime format (Important for forecasting later)
df['Date'] = pd.to_datetime(df['Date'])

# Show the first 5 rows
df.head()

In [ ]:
print("Missing values in each column:\n")
print(df.isnull().sum())
print(f"\nTotal rows in dataset: {len(df)}")

In [ ]:
# Since this is time-series data, if today's sensor reading is missing,
# it's best to assume it's similar to yesterday's reading.
# We use 'ffill' (forward fill) grouped by each City.

# 1. Sort by City and Date to ensure chronological order
df = df.sort_values(['City', 'Date'])

# 2. Forward fill the missing values city by city
cols_to_fill = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI']

for col in cols_to_fill:
    df[col] = df.groupby('City')[col].ffill()
    # If the very first day is missing, ffill won't work, so we backward fill (bfill) as a backup
    df[col] = df.groupby('City')[col].bfill()

# 3. Drop any rows that STILL have no AQI (if a city had no AQI sensors at all)
df = df.dropna(subset=['AQI'])

print("Missing values after cleaning:\n")
print(df.isnull().sum())